# VAI NVS Competition — Round 2 Pipeline

**7 scenes:** HCM0421, HCM0539, HCM0540, HCM0644, HCM0674, bonsai, chair

**GPU:** T4 x2 | **Est. time:** ~5.5h

Dataset Kaggle name: `vai-nvs-round2` (upload toàn bộ VAI_NVS_DATA_ROUND2/)

In [ ]:
# CELL 1: Kiểm tra GPU & Disk
import subprocess, os, shutil, time

print('=== GPU INFO ===')
subprocess.run(['nvidia-smi'], check=True)

print('\n=== DISK SPACE ===')
total, used, free = shutil.disk_usage('/kaggle/working')
print(f'Free: {free/1e9:.1f} GB / Total: {total/1e9:.1f} GB')

import torch
print(f'CUDA: {torch.cuda.is_available()}, GPUs: {torch.cuda.device_count()}')
for i in range(torch.cuda.device_count()):
    print(f'  GPU {i}: {torch.cuda.get_device_name(i)}')


In [ ]:
# CELL 2: Clone & Compile Gaussian Splatting
import subprocess, os

GS_DIR = '/kaggle/working/gaussian-splatting'

if not os.path.exists(GS_DIR):
    subprocess.run(['git','clone','--recursive',
        'https://github.com/graphdeco-inria/gaussian-splatting', GS_DIR], check=True)
else:
    print('Already cloned.')

subprocess.run(['pip','install','-q','plyfile','tqdm','lpips','scikit-image'], check=True)
subprocess.run(['pip','install','-q',f'{GS_DIR}/submodules/diff-gaussian-rasterization'], check=True)
subprocess.run(['pip','install','-q',f'{GS_DIR}/submodules/simple-knn'], check=True)
# fused-ssim: khong co tren PyPI, cai tu submodule da clone
try:
    subprocess.run(['pip','install','-q',f'{GS_DIR}/submodules/fused-ssim'], check=True)
    print('fused-ssim OK')
except Exception as e:
    print(f'fused-ssim skip: {e}')
print('\nSetup complete!')


In [ ]:
# ============================================================
# CELL 2.5: Apply Custom Patches to Gaussian Splatting Repo
# ============================================================
import subprocess, os, re

GS_DIR = '/kaggle/working/gaussian-splatting'

# Reset files to clean git state first to prevent corruption from multiple runs
print('Resetting modified files to clean git state...')
subprocess.run(['git', 'checkout', 'scene/dataset_readers.py'], cwd=GS_DIR)
subprocess.run(['git', 'checkout', 'utils/camera_utils.py'], cwd=GS_DIR)
subprocess.run(['git', 'checkout', 'scene/gaussian_model.py'], cwd=GS_DIR)

# == Patch 1: dataset_readers.py ==
dr_path = f'{GS_DIR}/scene/dataset_readers.py'
with open(dr_path) as f:
    code = f.read().replace('\r\n', '\n')

# 1a: Add SIMPLE_RADIAL support
OLD1 = '''        else:
            assert False, "Colmap camera model not handled: only undistorted datasets (PINHOLE or SIMPLE_PINHOLE cameras) supported!"'''
OLD1 = OLD1.replace('\r\n', '\n')

NEW1 = '''        elif intr.model in ("SIMPLE_RADIAL", "RADIAL"):
            focal_length_x = intr.params[0]
            FovY = focal2fov(focal_length_x, height)
            FovX = focal2fov(focal_length_x, width)
        elif intr.model == "OPENCV":
            focal_length_x = intr.params[0]
            focal_length_y = intr.params[1]
            FovY = focal2fov(focal_length_y, height)
            FovX = focal2fov(focal_length_x, width)
        else:
            raise ValueError(f"Unsupported camera model: {intr.model}")'''
NEW1 = NEW1.replace('\r\n', '\n')

if OLD1 in code:
    code = code.replace(OLD1, NEW1)
    print('Patch 1a applied: SIMPLE_RADIAL support')
else:
    print('Patch 1a: pattern not found, trying regex...')
    code = re.sub(
        r'else:\\s*\\n\\s*assert False, "Colmap camera model not handled.*?"',
        NEW1,
        code
    )

# 1b: Read-only filesystem fix for PLY
OLD2 = '''        storePly(ply_path, xyz, rgb)
    try:
        pcd = fetchPly(ply_path)'''
OLD2 = OLD2.replace('\r\n', '\n')

NEW2 = '''        try:
            storePly(ply_path, xyz, rgb)
        except OSError:
            import tempfile, hashlib
            _h = hashlib.md5(path.encode()).hexdigest()[:8]
            ply_path = f"/tmp/pts3d_{_h}.ply"
            storePly(ply_path, xyz, rgb)
    try:
        pcd = fetchPly(ply_path)'''
NEW2 = NEW2.replace('\r\n', '\n')

if OLD2 in code:
    code = code.replace(OLD2, NEW2)
    print('Patch 1b applied: read-only filesystem fix')
else:
    print('Patch 1b: pattern not found!')

# 1c: Optimize storePly memory usage
OLD1c = '''    normals = np.zeros_like(xyz)

    elements = np.empty(xyz.shape[0], dtype=dtype)
    attributes = np.concatenate((xyz, normals, rgb), axis=1)
    elements[:] = list(map(tuple, attributes))'''
OLD1c = OLD1c.replace('\r\n', '\n')

NEW1c = '''    elements = np.empty(xyz.shape[0], dtype=dtype)
    elements['x'] = xyz[:, 0]
    elements['y'] = xyz[:, 1]
    elements['z'] = xyz[:, 2]
    elements['nx'] = 0.0
    elements['ny'] = 0.0
    elements['nz'] = 0.0
    elements['red'] = rgb[:, 0]
    elements['green'] = rgb[:, 1]
    elements['blue'] = rgb[:, 2]'''
NEW1c = NEW1c.replace('\r\n', '\n')

if OLD1c in code:
    code = code.replace(OLD1c, NEW1c)
    print('Patch 1c applied: storePly memory optimization')
else:
    print('Patch 1c: pattern not found!')

with open(dr_path, 'w', newline='\n') as f:
    f.write(code)

# == Patch 2: camera_utils.py ==
cu_path = f'{GS_DIR}/utils/camera_utils.py'
with open(cu_path) as f:
    cu = f.read().replace('\r\n', '\n')

# 2a: Skip missing image files
OLD3 = '''    image = Image.open(cam_info.image_path)'''
OLD3 = OLD3.replace('\r\n', '\n')

NEW3 = '''    if not os.path.exists(cam_info.image_path):
        return None
    image = Image.open(cam_info.image_path)'''
NEW3 = NEW3.replace('\r\n', '\n')

if OLD3 in cu:
    cu = cu.replace(OLD3, NEW3)
    print('Patch 2a applied: skip missing images (loadCam)')
else:
    print('Patch 2a: pattern not found!')

if 'import os' not in cu.split('def ')[0]:
    cu = 'import os\n' + cu

# 2b: Filter None cameras
OLD4 = '''camera_list.append(loadCam(args, id, c, resolution_scale, is_nerf_synthetic, is_test_dataset))'''
OLD4 = OLD4.replace('\r\n', '\n')

NEW4 = '''_c = loadCam(args, id, c, resolution_scale, is_nerf_synthetic, is_test_dataset)
        if _c is not None:
            camera_list.append(_c)'''
NEW4 = NEW4.replace('\r\n', '\n')

if OLD4 in cu:
    cu = cu.replace(OLD4, NEW4)
    print('Patch 2b applied: filter None cameras (cameraList_from_camInfos)')
else:
    OLD4_alt = '''camera_list.append(loadCam(args, id, c, resolution_scale))'''
    OLD4_alt = OLD4_alt.replace('\r\n', '\n')
    
    NEW4_alt = '''_c = loadCam(args, id, c, resolution_scale)
        if _c is not None:
            camera_list.append(_c)'''
    NEW4_alt = NEW4_alt.replace('\r\n', '\n')
    
    if OLD4_alt in cu:
        cu = cu.replace(OLD4_alt, NEW4_alt)
        print('Patch 2b (alt) applied: filter None cameras (cameraList_from_camInfos)')
    else:
        print('Patch 2b: pattern not found!')

with open(cu_path, 'w', newline='\n') as f:
    f.write(cu)

# == Patch 3: gaussian_model.py ==
gm_path = f'{GS_DIR}/scene/gaussian_model.py'
with open(gm_path) as f:
    gm = f.read().replace('\r\n', '\n')

# 3a: Optimize save_ply memory usage
OLD5 = '''        elements = np.empty(xyz.shape[0], dtype=dtype_full)
        attributes = np.concatenate((xyz, normals, f_dc, f_rest, opacities, scale, rotation), axis=1)
        elements[:] = list(map(tuple, attributes))'''
OLD5 = OLD5.replace('\r\n', '\n')

NEW5 = '''        elements = np.empty(xyz.shape[0], dtype=dtype_full)
        elements['x'] = xyz[:, 0]
        elements['y'] = xyz[:, 1]
        elements['z'] = xyz[:, 2]
        elements['nx'] = normals[:, 0]
        elements['ny'] = normals[:, 1]
        elements['nz'] = normals[:, 2]
        for i in range(f_dc.shape[1]):
            elements[f'f_dc_{i}'] = f_dc[:, i]
        for i in range(f_rest.shape[1]):
            elements[f'f_rest_{i}'] = f_rest[:, i]
        elements['opacity'] = opacities[:, 0]
        for i in range(scale.shape[1]):
            elements[f'scale_{i}'] = scale[:, i]
        for i in range(rotation.shape[1]):
            elements[f'rot_{i}'] = rotation[:, i]'''
NEW5 = NEW5.replace('\r\n', '\n')

if OLD5 in gm:
    gm = gm.replace(OLD5, NEW5)
    print('Patch 3 applied: save_ply memory optimization')
else:
    print('Patch 3: pattern not found!')

with open(gm_path, 'w', newline='\n') as f:
    f.write(gm)

print()
print('All patches done!')
print('Data structure per scene:')
print('  images.bin references: ~371 images (original full set)')
print('  train/images/: 240 files  <- used for 3DGS training')
print('  test/images/:   60 files  <- in separate folder, will be skipped')
print('  Missing:        71 files  <- skipped automatically')


In [ ]:
# CELL 2.7: Blur Filtering — Lọc ảnh mờ trước training
# Phân tích: bonsai có 59/248 frames (24%) bị blur (Laplacian < 80)
# → 3DGS học ảnh mờ → Gaussians sai → PSNR/SSIM thấp
# Fix: tạo images_filtered/ chỉ chứa ảnh sắc nét
import cv2, shutil, os, time
from pathlib import Path

BLUR_THRESHOLD = 80   # Laplacian variance; <80 → mờ
FILTER_SCENES  = ['bonsai']  # chair chỉ 3% blur → bỏ qua để tiết kiệm thời gian
FILTERED_BASE  = '/kaggle/working/filtered'

t0 = time.time()
for scene in FILTER_SCENES:
    src_train = Path(f'{DATA_ROOT}/{scene}/train')
    dst_root  = Path(f'{FILTERED_BASE}/{scene}/train')
    dst_img   = dst_root / 'images'
    dst_img.mkdir(parents=True, exist_ok=True)

    # Symlink sparse/ (không cần copy binary files)
    sparse_dst = dst_root / 'sparse'
    if not sparse_dst.exists():
        os.symlink(str(src_train / 'sparse'), str(sparse_dst))

    kept, dropped = 0, 0
    all_imgs = sorted((src_train / 'images').glob('*.jpg'))
    for p in all_imgs:
        g = cv2.imread(str(p), cv2.IMREAD_GRAYSCALE)
        score = cv2.Laplacian(g, cv2.CV_64F).var() if g is not None else 0
        if score >= BLUR_THRESHOLD:
            shutil.copy2(p, dst_img / p.name)
            kept += 1
        else:
            dropped += 1

    pct = 100 * dropped / max(kept + dropped, 1)
    print(f'{scene}: kept={kept}/{kept+dropped} | dropped={dropped} ({pct:.1f}%) '
          f'| threshold={BLUR_THRESHOLD}')

    _, _, free = shutil.disk_usage('/kaggle/working')
    print(f'  Disk free: {free/1e9:.1f} GB')

print(f'Blur filtering done in {(time.time()-t0)/60:.1f} min')
print(f'FILTER_SCENES  = {FILTER_SCENES}')
print(f'FILTERED_BASE  = {FILTERED_BASE}')


In [ ]:
# CELL 3: Config Round 2 (v2 — Revised hyperparams based on experiments)
import os

DATA_ROOT     = '/kaggle/input/vai-nvs-round2'
OUT_DIR       = '/kaggle/working/output'
RENDER_DIR    = '/kaggle/working/submission'
FILTERED_BASE = '/kaggle/working/filtered'  # Blur-filtered train dirs
FILTER_SCENES = ['bonsai']                  # Scenes with blur filtering applied

ALL_SCENES = ['HCM0421', 'HCM0539', 'HCM0540', 'HCM0644', 'HCM0674', 'bonsai', 'chair']

# ─── Key lessons learned ─────────────────────────────────────────────────
# densify_grad LOWER → MORE Gaussians → overfitting → PSNR DROPS
#   0.0002 (default) >> 0.00008 (round2) >> 0.00005 (experiment, -0.66 PSNR)
# → Use 0.0002 default for all scenes
#
# bonsai:  SIMPLE_PINHOLE, 54K pts, 24% frames blurred → use filtered images
# chair:   SIMPLE_PINHOLE, 80K pts,  3% frames blurred → use as-is
# HCM*:    SIMPLE_RADIAL,  150-220K pts, 0% blur      → well-conditioned
# ─────────────────────────────────────────────────────────────────────────

SCENE_CONFIG = {
    # HCM drone scenes: large outdoor, SIMPLE_RADIAL, scale~20, no blur
    # densify_grad back to 0.0002 default; more iterations
    'HCM0421': {'resolution': 1, 'iterations': 35000,
                'densify_from': 500, 'densify_until': 30000,
                'densification_interval': 100,
                'densify_grad': 0.0002,
                'sh_degree': 3, 'percent_dense': 0.01,
                'opacity_reset_interval': 3000, 'data_device': 'cpu'},
    'HCM0539': {'resolution': 1, 'iterations': 35000,
                'densify_from': 500, 'densify_until': 30000,
                'densification_interval': 100,
                'densify_grad': 0.0002,
                'sh_degree': 3, 'percent_dense': 0.01,
                'opacity_reset_interval': 3000, 'data_device': 'cpu'},
    'HCM0540': {'resolution': 1, 'iterations': 35000,
                'densify_from': 500, 'densify_until': 30000,
                'densification_interval': 100,
                'densify_grad': 0.0002,
                'sh_degree': 3, 'percent_dense': 0.01,
                'opacity_reset_interval': 3000, 'data_device': 'cpu'},
    'HCM0644': {'resolution': 1, 'iterations': 35000,
                'densify_from': 500, 'densify_until': 30000,
                'densification_interval': 100,
                'densify_grad': 0.0002,
                'sh_degree': 3, 'percent_dense': 0.01,
                'opacity_reset_interval': 3000, 'data_device': 'cpu'},
    'HCM0674': {'resolution': 1, 'iterations': 35000,
                'densify_from': 500, 'densify_until': 30000,
                'densification_interval': 100,
                'densify_grad': 0.0002,
                'sh_degree': 3, 'percent_dense': 0.01,
                'opacity_reset_interval': 3000, 'data_device': 'cpu'},
    # bonsai: indoor, SIMPLE_PINHOLE, scale=7.3, 24% blur filtered
    # resolution=1 (scale_factor=1/1 → original 1920x1080, no downsample needed)
    'bonsai':  {'resolution': 1, 'iterations': 35000,
                'densify_from': 500, 'densify_until': 30000,
                'densification_interval': 100,
                'densify_grad': 0.0002,
                'sh_degree': 3, 'percent_dense': 0.01,
                'opacity_reset_interval': 3000, 'data_device': 'cpu'},
    # chair: indoor, SIMPLE_PINHOLE, scale=6.2, scale_factor=1/1.5
    # resolution=2 keeps images at 720x1280 (already scaled 1.5x down)
    'chair':   {'resolution': 2, 'iterations': 35000,
                'densify_from': 500, 'densify_until': 30000,
                'densification_interval': 100,
                'densify_grad': 0.0002,
                'sh_degree': 3, 'percent_dense': 0.01,
                'opacity_reset_interval': 3000, 'data_device': 'cpu'},
}

LAMBDA_DSSIM           = 0.2   # default — L1+SSIM balance
SH_DEGREE              = 3     # global fallback for render
DELETE_CKPT_AFTER_RENDER = True

# Indoor scenes use white background (wall/floor visible)
# Outdoor scenes (HCM*) use black background (sky)
INDOOR_SCENES = ['bonsai', 'chair']

print(f'DATA_ROOT : {DATA_ROOT}')
print(f'Total scenes: {len(ALL_SCENES)}')
for sc in ALL_SCENES:
    cfg = SCENE_CONFIG[sc]
    # Determine actual train path
    if sc in FILTER_SCENES:
        train_path = f'{FILTERED_BASE}/{sc}/train'
        path_tag = '[FILTERED]'
    else:
        train_path = f'{DATA_ROOT}/{sc}/train'
        path_tag = '[ORIGINAL]'
    status = 'OK' if os.path.exists(train_path) else 'NOT FOUND'
    print(f'  {sc:12s} res={cfg["resolution"]} iter={cfg["iterations"]:5d} '
          f'grad={cfg["densify_grad"]} sh={cfg["sh_degree"]} '
          f'{path_tag} — {status}')


In [ ]:
# CELL 4: Train Loop (v2)
import subprocess, time, os
from concurrent.futures import ThreadPoolExecutor, as_completed

session_start = time.time()
train_log = []

def get_ply_points_count(ply_path):
    if not os.path.exists(ply_path):
        return 0
    try:
        with open(ply_path, 'rb') as f:
            for _ in range(30):
                line = f.readline().decode('ascii', errors='ignore')
                if 'element vertex' in line:
                    return int(line.split()[-1])
                if 'end_header' in line:
                    break
    except Exception:
        pass
    return 0

def train_scene(args):
    gpu_id, scene = args
    cfg        = SCENE_CONFIG.get(scene, {})
    iterations = cfg.get('iterations', 35000)
    model_path = f'{OUT_DIR}/{scene}'
    log_path   = f'{model_path}/train.log'
    ply_path   = f'{model_path}/point_cloud/iteration_{iterations}/point_cloud.ply'

    # Use filtered images for blur-filtered scenes, original otherwise
    if scene in FILTER_SCENES:
        src_path = f'{FILTERED_BASE}/{scene}/train'
    else:
        src_path = f'{DATA_ROOT}/{scene}/train'

    if os.path.exists(ply_path):
        num_g = get_ply_points_count(ply_path)
        print(f'[GPU{gpu_id}] {scene}: checkpoint found ({num_g:,} Gaussians), skipping.', flush=True)
        return scene, 0, 'skipped', num_g

    os.makedirs(model_path, exist_ok=True)
    env = os.environ.copy()
    env['CUDA_VISIBLE_DEVICES'] = str(gpu_id)
    env['PYTORCH_CUDA_ALLOC_CONF'] = 'expandable_segments:True'
    env['PYTHONUNBUFFERED'] = '1'

    t0 = time.time()
    print(f'[GPU{gpu_id}] {scene}: starting (src={src_path}) (log: {log_path})', flush=True)

    with open(log_path, 'w', buffering=1) as lf:
        proc = subprocess.Popen([
            'python', f'{GS_DIR}/train.py',
            '-s', src_path, '-m', model_path,
            '--iterations',             str(iterations),
            '--sh_degree',              str(cfg.get('sh_degree', 3)),
            '--densify_from_iter',      str(cfg.get('densify_from', 500)),
            '--densify_until_iter',     str(cfg.get('densify_until', 15000)),
            '--densification_interval', str(cfg.get('densification_interval', 100)),
            '--densify_grad_threshold', str(cfg.get('densify_grad', 0.0002)),
            '--percent_dense',          str(cfg.get('percent_dense', 0.01)),
            '--opacity_reset_interval', str(cfg.get('opacity_reset_interval', 3000)),
            '--lambda_dssim',           str(LAMBDA_DSSIM),
            '--save_iterations',        str(iterations),
            '--test_iterations',        '-1',
            '--resolution',             str(cfg.get('resolution', 1)),
            '--position_lr_max_steps',  str(iterations),
            '--disable_viewer',
            '--data_device',            cfg.get('data_device', 'cpu'),
        ], stdout=lf, stderr=subprocess.STDOUT, env=env, text=True, bufsize=1)
        proc.wait()

    elapsed_min = (time.time() - t0) / 60
    if proc.returncode != 0:
        print(f'[GPU{gpu_id}] {scene} FAILED after {elapsed_min:.1f} min', flush=True)
        with open(log_path) as lf:
            print(lf.read()[-2000:], flush=True)
        return scene, elapsed_min, 'failed', 0

    num_g = get_ply_points_count(ply_path)
    print(f'[GPU{gpu_id}] {scene} DONE  {elapsed_min:.1f} min — Gaussians: {num_g:,}', flush=True)
    return scene, elapsed_min, 'ok', num_g

tasks = [(i % 2, sc) for i, sc in enumerate(ALL_SCENES)]
print(f'Running {len(tasks)} scenes on 2 GPUs...', flush=True)
print(f'Blur-filtered scenes: {FILTER_SCENES}', flush=True)

for idx in range(0, len(tasks), 2):
    batch     = tasks[idx: idx+2]
    batch_str = ' | '.join([f'GPU{t[0]}:{t[1]}' for t in batch])
    elapsed_h = (time.time() - session_start) / 3600
    print(f'\n--- Batch {idx//2+1}: {batch_str} [session={elapsed_h:.2f}h] ---', flush=True)

    with ThreadPoolExecutor(max_workers=2) as ex:
        futures = {ex.submit(train_scene, t): t for t in batch}
        for fut in as_completed(futures):
            res = fut.result()
            train_log.append(res)

print('\nTRAIN SUMMARY:')
for sc, t, s, n_g in train_log:
    print(f'  {sc:12s} - {s:8s} - {t:5.1f} min - Gaussians: {n_g:10,d}')


In [ ]:
# CELL 5: Render từ test_poses.csv (Round 2 v2)
import sys
sys.path.insert(0, GS_DIR)

import torch, numpy as np, pandas as pd, os, shutil, time
from PIL import Image
from gaussian_renderer import render
from scene import GaussianModel

def qvec2rotmat(qw, qx, qy, qz):
    return np.array([
        [1-2*(qy**2+qz**2),   2*(qx*qy-qz*qw),   2*(qx*qz+qy*qw)],
        [  2*(qx*qy+qz*qw), 1-2*(qx**2+qz**2),   2*(qy*qz-qx*qw)],
        [  2*(qx*qz-qy*qw),   2*(qy*qz+qx*qw), 1-2*(qx**2+qy**2)],
    ])

class SimplePipeline:
    convert_SHs_python   = False
    compute_cov3D_python = False
    debug                = False
    antialiasing         = True   # ON: smoother edges, better LPIPS

def render_scene(scene, model_path, poses_csv, output_dir, iterations):
    from scene.cameras import Camera
    os.makedirs(output_dir, exist_ok=True)

    ply_path = f'{model_path}/point_cloud/iteration_{iterations}/point_cloud.ply'
    if not os.path.exists(ply_path):
        print(f'  PLY not found: {ply_path}')
        return False, 0

    gaussians = GaussianModel(sh_degree=SH_DEGREE)
    gaussians.load_ply(ply_path)
    num_gaussians = len(gaussians.get_xyz)
    print(f'  Loaded {num_gaussians:,} Gaussians')

    # Background: white for indoor (bonsai/chair), black for outdoor (HCM* drone)
    if scene in INDOOR_SCENES:
        bg = torch.tensor([1., 1., 1.], dtype=torch.float32, device='cuda')
        print(f'  Background: WHITE (indoor scene)')
    else:
        bg = torch.tensor([0., 0., 0.], dtype=torch.float32, device='cuda')
        print(f'  Background: BLACK (outdoor scene)')

    pipe = SimplePipeline()
    df   = pd.read_csv(poses_csv)

    for idx, row in df.iterrows():
        W, H = int(row['width']), int(row['height'])
        R_w2c = qvec2rotmat(row['qw'], row['qx'], row['qy'], row['qz'])
        R = np.transpose(R_w2c)
        T = np.array([row['tx'], row['ty'], row['tz']])
        FoVx = 2 * np.arctan(W / (2 * row['fx']))
        FoVy = 2 * np.arctan(H / (2 * row['fy']))
        dummy_pil = Image.fromarray(np.zeros((H, W, 3), dtype=np.uint8))

        cam = Camera(
            resolution=(W, H), colmap_id=idx, R=R, T=T,
            FoVx=FoVx, FoVy=FoVy,
            depth_params=None, image=dummy_pil, invdepthmap=None,
            image_name=row['image_name'], uid=idx,
            data_device='cuda', train_test_exp=False,
            is_test_dataset=False, is_test_view=False,
        )

        with torch.no_grad():
            pkg = render(cam, gaussians, pipe, bg)

        img_np  = (pkg['render'].clamp(0, 1).permute(1, 2, 0).cpu().numpy() * 255).astype(np.uint8)
        img_pil = Image.fromarray(img_np)
        if img_pil.size != (W, H):
            img_pil = img_pil.resize((W, H), Image.LANCZOS)

        out_name = row['image_name']
        out_ext  = os.path.splitext(out_name)[1].upper()
        out_path = os.path.join(output_dir, out_name)
        if out_ext in ('.JPG', '.JPEG'):
            img_pil.save(out_path, format='JPEG', quality=98, subsampling=0)
        else:
            img_pil.save(out_path, format='PNG', optimize=True, compress_level=6)

        if idx % 15 == 0:
            print(f'    [{idx+1}/{len(df)}] {row["image_name"]}')

    print(f'  Rendered {len(df)} images -> {output_dir} ({num_gaussians:,} Gaussians)')
    return True, num_gaussians

render_log = []
for scene in ALL_SCENES:
    cfg        = SCENE_CONFIG.get(scene, {})
    iterations = cfg.get('iterations', 35000)
    model_path = f'{OUT_DIR}/{scene}'
    poses_csv  = f'{DATA_ROOT}/{scene}/test/test_poses.csv'
    output_dir = f'{RENDER_DIR}/{scene}'

    print(f'\n--- Rendering: {scene} ---')
    t0 = time.time()
    ok, n_g = render_scene(scene, model_path, poses_csv, output_dir, iterations)
    elapsed = (time.time() - t0) / 60
    render_log.append((scene, elapsed, 'ok' if ok else 'failed', n_g))

    if ok and DELETE_CKPT_AFTER_RENDER:
        ckpt = f'{model_path}/point_cloud'
        if os.path.exists(ckpt):
            shutil.rmtree(ckpt)
            print(f'  Checkpoint deleted.')

    _, _, free = shutil.disk_usage('/kaggle/working')
    print(f'  Disk free: {free/1e9:.1f} GB')

print('\nRENDER SUMMARY:')
for sc, t, s, n_g in render_log:
    print(f'  {sc:12s} - {s:8s} - {t:5.1f} min - Gaussians: {n_g:10,d}')


In [ ]:
# CELL 6: Evaluate (nếu có ground-truth trong test/images/)
import torch, lpips, numpy as np, os, glob
from PIL import Image
from skimage.metrics import structural_similarity as calc_ssim
from skimage.metrics import peak_signal_noise_ratio as calc_psnr

# Chỉ các scene có test/images/ gt
EVAL_SCENES = [sc for sc in ALL_SCENES
               if os.path.exists(f'{DATA_ROOT}/{sc}/test/images')]

if not EVAL_SCENES:
    print('Không tìm thấy gt images trong test/ — bỏ qua eval.')
else:
    loss_fn = lpips.LPIPS(net='alex').cuda()
    PSNR_MAX = 50.0  # Competition uses 50 (confirmed from score reverse-engineering)
    all_scores = []

    def to_tensor(img_np):
        t = torch.from_numpy(img_np).float() / 255.0
        return t.permute(2,0,1).unsqueeze(0).cuda() * 2 - 1

    for scene in EVAL_SCENES:
        gt_dir   = f'{DATA_ROOT}/{scene}/test/images'
        pred_dir = f'{RENDER_DIR}/{scene}'
        if not os.path.exists(pred_dir):
            print(f'⚠️  {scene}: pred dir missing'); continue

        lv, sv, pv = [], [], []
        for gt_path in sorted(glob.glob(f'{gt_dir}/*')):
            base = os.path.basename(gt_path)
            pred_path = os.path.join(pred_dir, base)
            if not os.path.exists(pred_path): continue
            gt_np   = np.array(Image.open(gt_path).convert('RGB'))
            pred_np = np.array(Image.open(pred_path).convert('RGB'))
            if gt_np.shape != pred_np.shape:
                pred_np = np.array(Image.fromarray(pred_np).resize(
                    (gt_np.shape[1], gt_np.shape[0]), Image.LANCZOS))
            with torch.no_grad():
                lv.append(loss_fn(to_tensor(gt_np), to_tensor(pred_np)).item())
            sv.append(calc_ssim(gt_np, pred_np, channel_axis=2, data_range=255))
            pv.append(calc_psnr(gt_np, pred_np, data_range=255))

        if not lv: print(f'⚠️  {scene}: no matched images'); continue
        alp, ass, aps = np.mean(lv), np.mean(sv), np.mean(pv)
        score = 0.4*(1-alp) + 0.3*ass + 0.3*min(aps/PSNR_MAX, 1.0)
        all_scores.append(score)
        print(f'{scene:12s} | LPIPS={alp:.4f} | SSIM={ass:.4f} | PSNR={aps:.2f}dB | Score={score:.4f}')

    if all_scores:
        print(f'\n★ Mean Score: {np.mean(all_scores):.4f}')


In [ ]:
# CELL 7: Validate & Package submission_round2.zip
import zipfile, glob, os, pandas as pd

ZIP_PATH = '/kaggle/working/submission_round2.zip'
errors   = []

print('=== Validating renders ===')
for scene in ALL_SCENES:
    poses_csv  = f'{DATA_ROOT}/{scene}/test/test_poses.csv'
    render_dir = f'{RENDER_DIR}/{scene}'
    df = pd.read_csv(poses_csv)
    expected_names = set(df['image_name'].tolist())

    if not os.path.exists(render_dir):
        errors.append(f'❌ {scene}: render dir missing'); continue

    found = set(os.path.basename(p) for p in
        glob.glob(f'{render_dir}/*.*'))
    missing = expected_names - found
    status = '✅' if not missing else f'⚠️ {len(found)}/{len(expected_names)}'
    print(f'  {scene:12s}: {status}')
    if missing:
        errors.append(f'{scene}: missing {len(missing)} images')

if errors:
    print('\nErrors:')
    for e in errors: print(f'  {e}')

print('\n=== Creating submission_round2.zip ===')
with zipfile.ZipFile(ZIP_PATH, 'w', zipfile.ZIP_STORED) as zf:
    for scene in ALL_SCENES:
        render_dir = f'{RENDER_DIR}/{scene}'
        if not os.path.exists(render_dir):
            print(f'  ⚠️ {scene}: skipped'); continue
        imgs = sorted(glob.glob(f'{render_dir}/*.*'))
        for img_path in imgs:
            zf.write(img_path, f'{scene}/{os.path.basename(img_path)}')
        print(f'  {scene}: {len(imgs)} images')

size_mb = os.path.getsize(ZIP_PATH) / 1e6
print(f'\n✅ submission_round2.zip: {size_mb:.1f} MB')
if size_mb > 500:
    print('⚠️  WARNING: > 500 MB limit!')
else:
    print('✅ Size OK (< 500 MB)')
print('→ Download từ Kaggle Output panel rồi submit!')
